In [1]:
import os
import sys
sys.path.append('/home/royhirsch/conformal/')

from ml_collections import config_dict
import logging
import pandas as pd
import pickle
import torch
import torch.nn as nn
import numpy as np

from conformal import get_conformal_module, get_percentile, clip_scores, calibrate_residual
from conformal_baselines import calc_baseline_mets
from data import get_dataloaders
from model import NN
from trainer import Trainer, get_optimizer, get_scheduler
from config import get_config_by_name
import utils as utils


In [2]:
config = get_config_by_name('tissuemnist')
config.conformal_module_name = 'aps'
config.num_epochs = 100
config.use_score_clipping = False 
config.plat_scaling = True


In [3]:
utils.seed_everything(config.seed)
utils.create_logger(config.exp_dir, config.dump_log)

logging.info('Config:')
for k, v in config.items():
    logging.info(f'{k}: {v}')
logging.info('')

conformal_module = get_conformal_module(config.conformal_module_name)
dls, t = get_dataloaders(config, conformal_module)
train_dl = dls['train']
valid_dl = dls['valid']
test_dl = dls['test']
baseline_mets = calc_baseline_mets(valid_dl, test_dl, alpha=config.alpha, k_raps=config.k_raps)

def split_by_thresh(dl, thresh, fold_name, batch_size=config.batch_size):
    ds = dl.dataset
    probs = ds.cls_probs.numpy()

    above_inds = np.where(probs.max(1) >= thresh)[0]
    below_inds = np.where(probs.max(1) < thresh)[0]
    above_ds = torch.utils.data.Subset(ds, above_inds)
    below_ds = torch.utils.data.Subset(ds, below_inds)
    print(f'Split with thresh: {thresh} to {len(above_ds)}/{len(below_ds)}')
    above_ds = torch.utils.data.DataLoader(above_ds, batch_size=batch_size, shuffle=True if fold_name == 'train' else False, pin_memory=True)
    below_ds = torch.utils.data.DataLoader(below_ds, batch_size=batch_size, shuffle=True if fold_name == 'train' else False, pin_memory=True)
    return above_ds, below_ds

thresh = 1. - config.alpha
above_train_dl, below_train_dl = split_by_thresh(train_dl, thresh, 'train')
above_valid_dl, below_valid_dl = split_by_thresh(valid_dl, thresh, 'valid')
above_test_dl, below_test_dl = split_by_thresh(test_dl, thresh, 'test')

below_baseline_mets = calc_baseline_mets(below_valid_dl, below_test_dl, alpha=config.alpha, k_raps=config.k_raps)
above_baseline_mets = calc_baseline_mets(above_valid_dl, above_test_dl, alpha=config.alpha, k_raps=config.k_raps)


INFO - 03/04/24 16:19:39 - 0:00:00 - Created main log at tmp/net_launcher_log.log
INFO - 03/04/24 16:19:39 - 0:00:00 - Config:
INFO - 03/04/24 16:19:39 - 0:00:00 - alpha: 0.1
INFO - 03/04/24 16:19:39 - 0:00:00 - batch_size: 128
INFO - 03/04/24 16:19:39 - 0:00:00 - comments: 
INFO - 03/04/24 16:19:39 - 0:00:00 - conformal_module_name: aps
INFO - 03/04/24 16:19:39 - 0:00:00 - criteria_name: mse
INFO - 03/04/24 16:19:39 - 0:00:00 - dataset_name: tissuemnist
INFO - 03/04/24 16:19:39 - 0:00:00 - device: cuda:0
INFO - 03/04/24 16:19:39 - 0:00:00 - drop_rate: 0.0
INFO - 03/04/24 16:19:39 - 0:00:00 - dump_log: False
INFO - 03/04/24 16:19:39 - 0:00:00 - exp_dir: tmp
INFO - 03/04/24 16:19:39 - 0:00:00 - file_name: /home/royhirsch/conformal/data/embeds_n_logits/aug/medmnist/tissuemnist_test.pickle
INFO - 03/04/24 16:19:39 - 0:00:00 - gpu_num: 0
INFO - 03/04/24 16:19:39 - 0:00:00 - hidden_dim: 512
INFO - 03/04/24 16:19:39 - 0:00:00 - input_dim: 2048
INFO - 03/04/24 16:19:39 - 0:00:00 - k_raps: 5
I

Split with thresh: 0.9 to 8879/30836
Split with thresh: 0.9 to 834/2948
Split with thresh: 0.9 to 881/2901


In [7]:
import numpy as np
import xgboost as xgb
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

params = {
    'objective': 'reg:squarederror',
    'tree_method': 'gpu_hist',  # Use GPU acceleration
    'gpu_id': 0,  # Specify GPU device index
    'max_depth': 3,
    'learning_rate': 0.08,
    'booster': 'dart',
    'n_estimators': 1000
}


def train()
dtrain = xgb.DMatrix(train_dl.dataset.embeds.numpy(), label=train_dl.dataset.scores.numpy())
dvalid = xgb.DMatrix(valid_dl.dataset.embeds.numpy(), label=valid_dl.dataset.scores.numpy())
dtest = xgb.DMatrix(test_dl.dataset.embeds.numpy(), label=test_dl.dataset.scores.numpy())

# Specify the parameters for XGBoost

# Train the XGBoost model
model = xgb.train(params, dtrain, num_boost_round=200)

mse = mean_squared_error(train_dl.dataset.scores.numpy(), model.predict(dtrain))
print(f"Mean Squared Error: {mse}")

mse = mean_squared_error(valid_dl.dataset.scores.numpy(), model.predict(dvalid))
print(f"Mean Squared Error: {mse}")

train_predict_out = {'true_scores': valid_dl.dataset.scores.numpy(),
                     'pred_scores': model.predict(dvalid),
                     'cls_probs': valid_dl.dataset.cls_probs.numpy(),
                     'cls_labels': valid_dl.dataset.cls_labels.numpy()}

test_predict_out = {'true_scores': test_dl.dataset.scores.numpy(),
                     'pred_scores': model.predict(dtest),
                     'cls_probs': test_dl.dataset.cls_probs.numpy(),
                     'cls_labels': test_dl.dataset.cls_labels.numpy()}

Mean Squared Error: 0.02046620100736618
Mean Squared Error: 0.021338624879717827


In [8]:
import numpy as np

alpha = 0.1
eps = 1e-6
max_value = 1.0 - eps

def quantize(preds, probs):
    quantize_scores = []
    for prob, pred in zip(probs, preds):
        cumsum = np.sort(prob)[::-1].cumsum()
        ind = np.argmin(np.abs(cumsum - pred))
        quantize_scores.append(cumsum[ind])
    return np.asarray(quantize_scores)

def calc_mets(probs, scores, labels):
    m = len(probs)
    val_pi = probs.argsort(1)[:, ::-1]
    val_srt = np.take_along_axis(probs, val_pi, axis=1).cumsum(axis=1)
    prediction_sets = np.take_along_axis(val_srt <= np.expand_dims(scores, 1), val_pi.argsort(axis=1), axis=1)

    print('mean size {:.3f} | acc {:.3f}'.format(prediction_sets.sum(1).mean(), 
                                                prediction_sets[np.arange(m), labels].mean()))
    return prediction_sets.sum(1).mean()

max_value = 1
alpha = 0.1
eps = 1e-12

calib_true_scores = train_predict_out['true_scores']
calib_pred_scores = train_predict_out['pred_scores']
calib_probs = train_predict_out['cls_probs']

valid_true_scores = test_predict_out['true_scores']
valid_true_labels = test_predict_out['cls_labels']
valid_pred_scores = test_predict_out['pred_scores']
valid_probs = test_predict_out['cls_probs']

calib_pred_scores = np.minimum(calib_pred_scores, max_value - eps)
valid_pred_scores = np.minimum(valid_pred_scores, max_value - eps)
calib_pred_scores = quantize(calib_pred_scores, calib_probs)
valid_pred_scores = quantize(valid_pred_scores, valid_probs)

diff = (calib_true_scores - calib_pred_scores) / (max_value - calib_probs.max(1) + eps)
# diff = calib_true_scores - calib_pred_scores

# diff = diff[np.where(calib_pred_scores<global_qhat)[0]]
n = len(diff)
qhat = np.quantile(diff, np.ceil((n + 1) * (1 - alpha)) / n)

modified_valid_pred_scores = valid_pred_scores + qhat * (max_value - valid_probs.max(1) + eps)
# modified_valid_pred_scores = valid_pred_scores + qhat

n_over_one = (modified_valid_pred_scores >= 1.).sum() / len(modified_valid_pred_scores)
print('Residuals correction is {:.8f}, clip {:.2f}% of the samples'.format(
    qhat, n_over_one * 100.))

modified_valid_pred_scores = np.minimum(modified_valid_pred_scores, max_value - eps)
calc_mets(valid_probs, modified_valid_pred_scores, test_predict_out['cls_labels'])


Residuals correction is 0.51579541, clip 31.76% of the samples
mean size 3.905 | acc 0.906


3.9048122686409306